# STT Arena Prediction Runner
## NGI0 Commons Fund deliverable — WP4

The OVOS STT arena defines a contract for submitting STT engine predictions
against a standardised test corpus (§3.2 of the arena specification).
This notebook demonstrates:

1. Producing a **20-sample prediction slice** in the arena §3.2 contract format
2. Using **edge-tts** to generate reference audio (circumventing the need for
   pre-recorded corpus audio in CI)
3. Transcribing with a local mock engine that follows the real schema —
   plus a live `fasterwhisper` path if the library is available

**Why this matters for NGI0:** the arena infrastructure enables reproducible
benchmarking of community-contributed STT plugins — a key goal of the
OpenVoiceOS accessibility work funded by NGI0.

> Developed by TigreGotico for OpenVoiceOS, funded by the
> [NGI0 Commons Fund](https://nlnet.nl/project/OpenVoiceOS) / NLnet grant **101135429**.

**CI execution:** ✅ executed headlessly.  GPU-heavy engines are skipped automatically.


## 0 · Configuration

In [1]:
import nest_asyncio
nest_asyncio.apply()

# 20 short sentences to transcribe
CORPUS = [
    "the weather is nice today",
    "please turn off the lights",
    "set a timer for five minutes",
    "what time is it",
    "play some jazz music",
    "how many days until christmas",
    "remind me to call the doctor",
    "what is the capital of portugal",
    "tell me a joke",
    "stop the alarm",
    "turn up the volume",
    "send a message to alice",
    "navigate to the nearest pharmacy",
    "open the front door",
    "is it going to rain tomorrow",
    "add milk to the shopping list",
    "what movies are showing tonight",
    "turn on the heating",
    "pause the podcast",
    "good night mycroft",
]

LANG = "en-us"
TTS_VOICE = "en-US-JennyNeural"   # consistent voice for reproducibility

print(f"Corpus size : {len(CORPUS)} sentences")
print(f"Language    : {LANG}")
print(f"TTS voice   : {TTS_VOICE}")


Corpus size : 20 sentences
Language    : en-us
TTS voice   : en-US-JennyNeural


## 1 · Generate reference audio

In [2]:
import asyncio, subprocess, tempfile, sys
from pathlib import Path

WORKDIR = Path(tempfile.mkdtemp(prefix="arena_"))
AUDIO_DIR = WORKDIR / "audio"
AUDIO_DIR.mkdir(parents=True)

try:
    import edge_tts
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "edge-tts"])
    import edge_tts

async def synth(text: str, voice: str, out_mp3: Path):
    comm = edge_tts.Communicate(text, voice)
    await comm.save(str(out_mp3))

wav_files = []
for i, sentence in enumerate(CORPUS):
    mp3 = AUDIO_DIR / f"sample_{i:02d}.mp3"
    wav = AUDIO_DIR / f"sample_{i:02d}.wav"
    asyncio.run(synth(sentence, TTS_VOICE, mp3))
    subprocess.run(
        ["ffmpeg", "-y", "-i", str(mp3), "-ar", "16000", "-ac", "1", str(wav)],
        check=True, capture_output=True,
    )
    mp3.unlink()
    wav_files.append(wav)
    if i % 5 == 4:
        print(f"  synthesised {i+1}/{len(CORPUS)} samples...")

print(f"\nAll {len(wav_files)} reference WAV files ready.")


  synthesised 5/20 samples...


  synthesised 10/20 samples...


  synthesised 15/20 samples...


  synthesised 20/20 samples...

All 20 reference WAV files ready.


## 2 · Mock STT engine (schema-compliant)

The mock engine returns the known ground-truth text, demonstrating the arena
data contract independently of any real ASR model.  The schema is used by the
real arena to collect predictions from community-contributed STT plugins.

A live `faster-whisper` path is tried if the library is available (falls back to mock).


In [3]:
import time, json, importlib

def mock_transcribe(wav_path: Path, lang: str = "en-us") -> dict:
    """Return ground-truth text — simulates a perfect STT engine."""
    idx = int(wav_path.stem.split("_")[1])
    return {
        "transcript": CORPUS[idx],
        "confidence": 1.0,
        "engine": "mock-ground-truth",
        "lang": lang,
    }

# Try faster-whisper if available
HAS_WHISPER = False
try:
    from faster_whisper import WhisperModel
    HAS_WHISPER = True
    print("faster-whisper available — will use for transcription")
except ImportError:
    print("faster-whisper not available — using mock engine")


faster-whisper available — will use for transcription


## 3 · Produce the arena §3.2 prediction slice

The arena contract requires a JSON Lines file where each row has:
```json
{"id": "sample_00", "hypothesis": "...", "reference": "...", "lang": "en-us", "engine": "..."}
```


In [4]:
if HAS_WHISPER:
    model = WhisperModel("tiny", device="cpu", compute_type="int8")
    print("WhisperModel loaded (tiny / int8 / CPU)")

predictions = []
t0 = time.time()

for i, (wav_path, reference) in enumerate(zip(wav_files, CORPUS)):
    if HAS_WHISPER:
        segments, info = model.transcribe(str(wav_path), language="en", beam_size=1)
        hypothesis = " ".join(seg.text.strip() for seg in segments)
        engine = f"faster-whisper-tiny"
    else:
        result = mock_transcribe(wav_path, LANG)
        hypothesis = result["transcript"]
        engine = result["engine"]

    predictions.append({
        "id": wav_path.stem,
        "hypothesis": hypothesis,
        "reference": reference,
        "lang": LANG,
        "engine": engine,
    })

elapsed = time.time() - t0
print(f"Transcribed {len(predictions)} samples in {elapsed:.1f}s")
print(f"Engine: {predictions[0]['engine']}")


WhisperModel loaded (tiny / int8 / CPU)


Transcribed 20 samples in 13.0s
Engine: faster-whisper-tiny


## 4 · Write JSONL output and compute WER

In [5]:
# Write JSONL
jsonl_path = WORKDIR / "predictions.jsonl"
with open(jsonl_path, "w") as f:
    for row in predictions:
        f.write(json.dumps(row) + "\n")

print(f"Written: {jsonl_path}")

# Simple WER computation
def word_error_rate(hyp: str, ref: str) -> float:
    h = hyp.lower().split()
    r = ref.lower().split()
    # Levenshtein distance (DP)
    dp = list(range(len(r) + 1))
    for i, hw in enumerate(h):
        ndp = [i + 1]
        for j, rw in enumerate(r):
            ndp.append(min(dp[j] + (0 if hw == rw else 1),
                           dp[j+1] + 1, ndp[j] + 1))
        dp = ndp
    return dp[len(r)] / max(len(r), 1)

wers = [word_error_rate(p["hypothesis"], p["reference"]) for p in predictions]
avg_wer = sum(wers) / len(wers)

print(f"\nWER summary (n={len(predictions)}):")
print(f"  Mean WER : {avg_wer:.4f}")
print(f"  Min WER  : {min(wers):.4f}")
print(f"  Max WER  : {max(wers):.4f}")

print("\nSample predictions:")
for p, w in zip(predictions[:5], wers[:5]):
    match = "✅" if w == 0.0 else "⚠"
    print(f"  {match} [{p['id']}] hyp={p['hypothesis']!r}  wer={w:.3f}")


Written: /tmp/arena_pjl0r4g0/predictions.jsonl

WER summary (n=20):
  Mean WER : 0.2683
  Min WER  : 0.1667
  Max WER  : 1.0000

Sample predictions:
  ⚠ [sample_00] hyp='The weather is nice today.'  wer=0.200
  ⚠ [sample_01] hyp='Please turn off the lights.'  wer=0.200
  ⚠ [sample_02] hyp='Set a timer for 5 minutes.'  wer=0.333
  ⚠ [sample_03] hyp='What time is it?'  wer=0.250
  ⚠ [sample_04] hyp='Play some jazz music.'  wer=0.250


## 5 · Arena contract summary

| Field | Value |
|---|---|
| Format | JSON Lines (one prediction per line) |
| Required keys | `id`, `hypothesis`, `reference`, `lang`, `engine` |
| Corpus size | 20 samples |
| Engine (CI) | mock-ground-truth (or faster-whisper-tiny if installed) |
| Mean WER (mock) | 0.0000 (ground-truth engine) |

**How the real arena uses this:**
1. A plugin submits a `predictions.jsonl` via a PR to the arena repo
2. CI computes WER / CER against the held-out reference corpus
3. Results are published to the leaderboard dashboard

This notebook demonstrates the submission contract end-to-end.


In [6]:
import shutil
shutil.rmtree(WORKDIR, ignore_errors=True)
print("Workdir cleaned up.")


Workdir cleaned up.
